<a href="https://colab.research.google.com/github/krisnapriyaa/Netflix-Movies-and-TV-Shows-EDA/blob/main/Netflix_Interactive_Dashboard.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

Install and import libraries

In [1]:
!pip install plotly -q
import pandas as pd
import numpy as np
import plotly.express as px
import plotly.graph_objects as go
from google.colab import files
print("Libraries imported successfully!")

Libraries imported successfully!


Load the dataset

In [2]:
uploaded = files.upload()
df = pd.read_csv('netflix_titles.csv')
print("Dataset loaded successfully!")
print("Shape:", df.shape)
df.head()

Saving netflix_titles.csv to netflix_titles.csv
Dataset loaded successfully!
Shape: (8807, 12)


,show_id,type,title,director,cast,country,date_added,release_year,rating,duration,listed_in,description
0,s1,Movie,Dick Johnson Is Dead,Kirsten Johnson,NaN,United States,"September 25, 2021",2020,PG-13,90 min,Documentaries,"As her father nears the end of his life, filmm..."
1,s2,TV Show,Blood & Water,NaN,"Ama Qamata, Khosi Ngema, Gail Mabalane, Thaban...",South Africa,"September 24, 2021",2021,TV-MA,2 Seasons,"International TV Shows, TV Dramas, TV Mysteries","After crossing paths at a party, a Cape Town t..."
2,s3,TV Show,Ganglands,Julien Leclercq,"Sami Bouajila, Tracy Gotoas, Samuel Jouy, Nabi...",NaN,"September 24, 2021",2021,TV-MA,1 Season,"Crime TV Shows, International TV Shows, TV Act...",To protect his family from a powerful drug lor...
3,s4,TV Show,Jailbirds New Orleans,NaN,NaN,NaN,"September 24, 2021",2021,TV-MA,1 Season,"Docuseries, Reality TV","Feuds, flirtations and toilet talk go down amo..."
4,s5,TV Show,Kota Factory,NaN,"Mayur More, Jitendra Kumar, Ranjan Raj, Alam K...",India,"September 24, 2021",2021,TV-MA,2 Seasons,"International TV Shows, Romantic TV Shows, TV ...",In a city of coaching centers known to train I...


Basic data preparation

In [3]:
# Create a copy
data = df.copy()

# Remove duplicate records
data = data.drop_duplicates()

# Clean text columns
data['type'] = data['type'].str.strip()
data['country'] = data['country'].fillna('Unknown')
data['rating'] = data['rating'].fillna('Unknown')

# Extract movie duration
data['movie_duration'] = pd.to_numeric(
    data['duration'].str.extract(r'^(\d+)\s*min$', expand=False),
    errors='coerce'
)

# Extract TV show seasons
data['seasons'] = pd.to_numeric(
    data['duration'].str.extract(r'^(\d+)\s*Seasons?$', expand=False),
    errors='coerce'
)

# Convert date_added to datetime
data['date_added'] = pd.to_datetime(
    data['date_added'].str.strip(),
    errors='coerce'
)

# Extract year added
data['year_added'] = data['date_added'].dt.year
print("Data preparation completed!")
print(data.shape)


Data preparation completed!
(8807, 15)


Interactive bar chart

In [4]:
content_counts = data['type'].value_counts().reset_index()
content_counts.columns = ['Content Type', 'Number of Titles']
fig = px.bar(
    content_counts,
    x='Content Type',
    y='Number of Titles',
    color='Content Type',
    title='Netflix Content Distribution',
    text='Number of Titles',
    template='plotly_white',
    color_discrete_sequence=['#E50914', '#564D4D']
)
fig.update_layout(
    title_x=0.5,
    xaxis_title='Content Type',
    yaxis_title='Number of Titles',
    font=dict(size=13)
)

fig.show()

Interactive Line Chart

In [5]:
# Count titles by release year and content type
yearly_data = (
    data.groupby(['release_year', 'type'])
    .size()
    .reset_index(name='Number of Titles')
)

# Create interactive line chart
fig = px.line(
    yearly_data,
    x='release_year',
    y='Number of Titles',
    color='type',
    markers=True,
    title='Netflix Content Release Trends Over the Years',
    labels={
        'release_year': 'Release Year',
        'Number of Titles': 'Number of Titles',
        'type': 'Content Type'
    },
    color_discrete_map={
        'Movie': '#E50914',
        'TV Show': '#564D4D'
    },
    template='plotly_white',
    hover_data={
        'release_year': True,
        'Number of Titles': True,
        'type': True
    }
)
fig.update_layout(
    title_x=0.5,
    font=dict(size=13),
    hovermode='x unified'
)

fig.show()

Prepare movie duration data

In [6]:
# Select only Movies
movies = data[data['type'] == 'Movie'].copy()

# Extract duration in minutes
movies['duration_minutes'] = pd.to_numeric(
    movies['duration'].str.extract(r'(\d+)')[0],
    errors='coerce'
)

# Remove missing duration values
movie_duration = movies.dropna(subset=['duration_minutes'])
print("Total Movies:", len(movie_duration))
print("\nDuration Statistics:")
print(movie_duration['duration_minutes'].describe())

Total Movies: 6128

Duration Statistics:
count    6128.000000
mean       99.577187
std        28.290593
min         3.000000
25%        87.000000
50%        98.000000
75%       114.000000
max       312.000000
Name: duration_minutes, dtype: float64


Create an interactive histogram

In [8]:
fig = px.histogram(
    movie_duration,
    x='duration_minutes',
    nbins=30,
    title='Distribution of Netflix Movie Durations',
    labels={
        'duration_minutes': 'Movie Duration (Minutes)',
        'count': 'Number of Movies'
    },
    color_discrete_sequence=['#E50914'],
    template='plotly_white',
    hover_data=['title']
)
fig.update_layout(
    title_x=0.5,
    xaxis_title='Movie Duration (Minutes)',
    yaxis_title='Number of Movies',
    font=dict(size=13),
    bargap=0
)

fig.show()

Calculate the average movie duration

In [9]:
print("Average Duration:",
      movie_duration['duration_minutes'].mean())
print("Median Duration:",
      movie_duration['duration_minutes'].median())
print("Shortest Movie:",
      movie_duration['duration_minutes'].min())
print("Longest Movie:",
      movie_duration['duration_minutes'].max())

Average Duration: 99.57718668407311
Median Duration: 98.0
Shortest Movie: 3.0
Longest Movie: 312.0


Prepare the data

In [10]:
# Select movies with valid durations and ratings
box_data = movie_duration.copy()

# Remove missing and invalid ratings
box_data = box_data[
    box_data['rating'].notna()
]
box_data = box_data[
    ~box_data['rating'].str.contains(
        r'^\d+\s*min$', case=False, regex=True
    )
]

# Select the 8 most common ratings
top_ratings = box_data['rating'].value_counts().head(8).index
box_data = box_data[
    box_data['rating'].isin(top_ratings)
]
print("Ratings selected:")
print(box_data['rating'].value_counts())

Ratings selected:
rating
TV-MA    2062
TV-14    1427
R         797
TV-PG     540
PG-13     490
PG        287
TV-Y7     139
TV-Y      131
Name: count, dtype: int64


Create the interactive box plot

In [11]:
fig = px.box(
    box_data,
    x='rating',
    y='duration_minutes',
    color='rating',
    title='Movie Duration Distribution by Content Rating',
    labels={
        'rating': 'Content Rating',
        'duration_minutes': 'Movie Duration (Minutes)'
    },
    points='outliers',
    template='plotly_white',
    color_discrete_sequence=px.colors.qualitative.Bold
)
fig.update_layout(
    title_x=0.5,
    xaxis_title='Content Rating',
    yaxis_title='Movie Duration (Minutes)',
    showlegend=False,
    font=dict(size=13)
)
fig.show()

Notebook interpretation

In [ ]:
'''The box plot compares movie durations across different content ratings.
 It illustrates the median duration, variability, and potential outliers for each rating category.
 This visualization helps identify differences in movie lengths across audience classifications.'''

Release Year vs Movie Duration

In [13]:
# Select movies with valid release year and duration
scatter_data = movie_duration.dropna(subset=['release_year', 'duration_minutes']).copy()
print("Total records:", len(scatter_data))
print(scatter_data[
    ['title', 'release_year', 'duration_minutes', 'rating']
].head())

Total records: 6128
                               title  release_year  duration_minutes rating
0               Dick Johnson Is Dead          2020              90.0  PG-13
6   My Little Pony: A New Generation          2021              91.0     PG
7                            Sankofa          1993             125.0  TV-MA
9                       The Starling          2021             104.0  PG-13
12                      Je Suis Karl          2021             127.0  TV-MA


Create an interactive scatter plot

In [14]:
fig = px.scatter(
    scatter_data,
    x='release_year',
    y='duration_minutes',
    color='type',
    hover_name='title',
    hover_data={
        'release_year': True,
        'duration_minutes': True,
        'rating': True
    },
    title='Relationship Between Movie Release Year and Duration',
    labels={
        'release_year': 'Release Year',
        'duration_minutes': 'Movie Duration (Minutes)',
        'type': 'Content Type'
    },
    color_discrete_sequence=['#E50914'],
    opacity=0.6,
    template='plotly_white'
)
fig.update_layout(
    title_x=0.5,
    xaxis_title='Release Year',
    yaxis_title='Movie Duration (Minutes)',
    font=dict(size=13)
)
fig.show()


'''The scatter plot explores the relationship between movie release years and their durations.
Each point represents an individual movie.
The visualization helps identify variations in movie length across different release years and highlights unusually short or long movies.
It also allows individual movie details to be explored interactively.'''

Netflix Genre Distribution (Treemap)

In [15]:
# Extract genre combinations
genres = data['listed_in'].dropna().str.split(',')

# Convert each genre into an individual row
genres = genres.explode().str.strip()

# Count genre appearances
genre_counts = genres.value_counts().reset_index()
genre_counts.columns = ['Genre', 'Number of Titles']

# Select top 15 genres
top_genres = genre_counts.head(15)
print(top_genres)

                       Genre  Number of Titles
0       International Movies              2752
1                     Dramas              2427
2                   Comedies              1674
3     International TV Shows              1351
4              Documentaries               869
5         Action & Adventure               859
6                  TV Dramas               763
7         Independent Movies               756
8   Children & Family Movies               641
9            Romantic Movies               616
10               TV Comedies               581
11                 Thrillers               577
12            Crime TV Shows               470
13                  Kids' TV               451
14                Docuseries               395


Create the interactive treemap

In [16]:
fig = px.treemap(
    top_genres,
    path=['Genre'],
    values='Number of Titles',
    title='Top 15 Netflix Genres Distribution',
    color='Number of Titles',
    color_continuous_scale='Reds',
    template='plotly_white'
)
fig.update_layout(
    title_x=0.5,
    font=dict(size=13)
)
fig.show()

'''The treemap visualizes the distribution of the most frequently occurring genres in Netflix's content library.
Larger rectangles represent genres with higher numbers of title appearances.
This visualization helps identify popular content categories and understand the diversity of Netflix's entertainment library.'''

the final interactive dashboard

In [17]:
import plotly.graph_objects as go
fig = go.Figure()
content_types = ['All', 'Movie', 'TV Show']
for content_type in content_types:
    if content_type == 'All':
        filtered = data
    else:
        filtered = data[data['type'] == content_type]
    counts = filtered['type'].value_counts()

    fig.add_trace(
        go.Bar(
            x=counts.index,
            y=counts.values,
            name=content_type,
            visible=(content_type == 'All')
        )
    )
fig.update_layout(
    title='Netflix Content Distribution Filter',
    template='plotly_white',
    updatemenus=[dict(
        buttons=[
            dict(
                label='All',
                method='update',
                args=[{'visible': [True, False, False]}]
            ),
            dict(
                label='Movies',
                method='update',
                args=[{'visible': [False, True, False]}]
            ),
            dict(
                label='TV Shows',
                method='update',
                args=[{'visible': [False, False, True]}]
            )
        ],
        direction='down',
        x=1.0,
        y=1.15
    )]
)
fig.show()

Export your interactive dashboard

In [19]:
fig.write_html(
    "Netflix_Content_Filter.html",
    include_plotlyjs=True
)
print("Interactive HTML exported successfully!")

Interactive HTML exported successfully!


The dashboard code

In [20]:
# ==========================================
# NETFLIX INTERACTIVE ANALYTICS DASHBOARD
# ==========================================

import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots

# 1. Prepare data
dashboard_data = data.copy()
dashboard_data['duration_minutes'] = pd.to_numeric(
    dashboard_data['duration'].str.extract(
        r'^(\d+)\s*min$', expand=False
    ),
    errors='coerce'
)
movies_data = dashboard_data[
    (dashboard_data['type'] == 'Movie') &
    (dashboard_data['duration_minutes'].notna())
].copy()

# 2. Create subplot layout
fig = make_subplots(
    rows=3,
    cols=2,
    specs=[
        [{"type": "xy"}, {"type": "xy"}],
        [{"type": "xy"}, {"type": "xy"}],
        [{"type": "xy"}, {"type": "domain"}]
    ],
    subplot_titles=[
        "Content Distribution",
        "Release Year Trends",
        "Movie Duration Distribution",
        "Duration by Content Rating",
        "Release Year vs Movie Duration",
        "Top Netflix Genres"
    ],
    vertical_spacing=0.13,
    horizontal_spacing=0.12
)

# 3. BAR CHART - Content distribution
content_types = ['All', 'Movie', 'TV Show']

for content_type in content_types:
    if content_type == 'All':
        filtered = dashboard_data
    else:
        filtered = dashboard_data[
            dashboard_data['type'] == content_type
        ]

    counts = filtered['type'].value_counts()

    fig.add_trace(
        go.Bar(
            x=counts.index,
            y=counts.values,
            name=content_type,
            marker_color=['#E50914', '#777777'],
            visible=(content_type == 'All'),
            text=counts.values,
            textposition='auto',
            hovertemplate=(
                'Type: %{x}<br>'
                'Titles: %{y}<extra></extra>'
            )
        ),
        row=1, col=1
    )

# 4. LINE CHART - Release trends
yearly = (
    dashboard_data.groupby(
        ['release_year', 'type']
    ).size().reset_index(name='Titles')
)

for content_type, color in [
    ('Movie', '#E50914'),
    ('TV Show', '#777777')
]:
    subset = yearly[yearly['type'] == content_type]

    fig.add_trace(
        go.Scatter(
            x=subset['release_year'],
            y=subset['Titles'],
            name=content_type,
            mode='lines+markers',
            line=dict(color=color, width=2),
            legendgroup=content_type,
            hovertemplate=(
                'Year: %{x}<br>'
                'Titles: %{y}<extra></extra>'
            )
        ),
        row=1, col=2
    )

# 5. HISTOGRAM - Movie duration
fig.add_trace(
    go.Histogram(
        x=movies_data['duration_minutes'],
        nbinsx=30,
        name='Movie Duration',
        marker_color='#E50914',
        hovertemplate=(
            'Duration range: %{x}<br>'
            'Movies: %{y}<extra></extra>'
        )
    ),
    row=2, col=1
)

# 6. BOX PLOT - Duration by rating
valid_ratings = movies_data[
    movies_data['rating'].notna()
].copy()

valid_ratings = valid_ratings[
    ~valid_ratings['rating'].str.contains(
        r'^\d+\s*min$', case=False, regex=True
    )
]

top_ratings = (
    valid_ratings['rating']
    .value_counts()
    .head(8).index
)

box_data = valid_ratings[
    valid_ratings['rating'].isin(top_ratings)
]

for rating in top_ratings:
    subset = box_data[box_data['rating'] == rating]

    fig.add_trace(
        go.Box(
            y=subset['duration_minutes'],
            name=rating,
            boxpoints='outliers',
            marker_color='#E50914',
            hovertemplate=(
                'Rating: ' + rating +
                '<br>Duration: %{y} min<extra></extra>'
            )
        ),
        row=2, col=2
    )

# 7. SCATTER PLOT - Release year vs duration
fig.add_trace(
    go.Scatter(
        x=movies_data['release_year'],
        y=movies_data['duration_minutes'],
        mode='markers',
        name='Movies',
        marker=dict(
            color='#E50914',
            opacity=0.55,
            size=7
        ),
        text=movies_data['title'],
        customdata=movies_data[['rating']],
        hovertemplate=(
            'Movie: %{text}<br>'
            'Year: %{x}<br>'
            'Duration: %{y} min<br>'
            'Rating: %{customdata[0]}'
            '<extra></extra>'
        )
    ),
    row=3, col=1
)

# 8. TREEMAP - Genre distribution
genres = (
    dashboard_data['listed_in']
    .dropna()
    .str.split(',')
    .explode()
    .str.strip()
)

genre_counts = genres.value_counts().head(15)

fig.add_trace(
    go.Treemap(
        labels=genre_counts.index,
        parents=[''] * len(genre_counts),
        values=genre_counts.values,
        marker=dict(
            colors=genre_counts.values,
            colorscale='Reds'
        ),
        name='Genres',
        hovertemplate=(
            'Genre: %{label}<br>'
            'Titles: %{value}<extra></extra>'
        )
    ),
    row=3, col=2
)

# 9. Dashboard styling
fig.update_layout(
    title=dict(
        text=(
            "NETFLIX ANALYTICS DASHBOARD"
            "<br><sup>Interactive Exploration of Movies and TV Shows</sup>"
        ),
        x=0.5,
        xanchor='center'
    ),
    template='plotly_white',
    height=1350,
    width=1200,
    font=dict(
        family='Arial',
        size=12,
        color='#222222'
    ),
    paper_bgcolor='white',
    plot_bgcolor='white',
    margin=dict(t=110, b=70, l=70, r=50),
    updatemenus=[
        dict(
            buttons=[
                dict(
                    label='All Content',
                    method='update',
                    args=[{
                        'visible': [
                            True, False, False,
                            True, True, True, True, True, True, True
                        ]
                    }]
                ),
                dict(
                    label='Movies',
                    method='update',
                    args=[{
                        'visible': [
                            False, True, False,
                            True, True, True, True, True, True, True
                        ]
                    }]
                ),
                dict(
                    label='TV Shows',
                    method='update',
                    args=[{
                        'visible': [
                            False, False, True,
                            True, True, True, True, True, True, True
                        ]
                    }]
                )
            ],
            direction='down',
            x=1,
            xanchor='right',
            y=1.10,
            yanchor='top'
        )
    ],
    annotations=list(fig.layout.annotations) + [
        dict(
            text='Filter Content Type:',
            x=0.78,
            y=1.10,
            xref='paper',
            yref='paper',
            showarrow=False
        )
    ]
)

# 10. Axis labels
fig.update_xaxes(title_text='Content Type', row=1, col=1)
fig.update_yaxes(title_text='Number of Titles', row=1, col=1)
fig.update_xaxes(title_text='Release Year', row=1, col=2)
fig.update_yaxes(title_text='Number of Titles', row=1, col=2)
fig.update_xaxes(title_text='Duration (Minutes)', row=2, col=1)
fig.update_yaxes(title_text='Number of Movies', row=2, col=1)
fig.update_xaxes(title_text='Content Rating', row=2, col=2)
fig.update_yaxes(title_text='Duration (Minutes)', row=2, col=2)
fig.update_xaxes(title_text='Release Year', row=3, col=1)
fig.update_yaxes(title_text='Duration (Minutes)', row=3, col=1)

# 11. Display dashboard
fig.show()
print("Netflix Interactive Dashboard created!")

Netflix Interactive Dashboard created!


dropdown correction

In [25]:
 # Calculate correct trace visibility
other_traces = 2 + 1 + len(top_ratings) + 1 + 1
all_content = [True, False, False] + [True] * other_traces
movies_only = [False, True, False] + [True] * other_traces
tv_only = [False, False, True] + [True] * other_traces

updatemenus=[
    dict(
        buttons=[
            dict(label='All Content', method='update',
                 args=[{'visible': all_content}]),
            dict(label='Movies', method='update',
                 args=[{'visible': movies_only}]),
            dict(label='TV Shows', method='update',
                 args=[{'visible': tv_only}])
        ],
        direction='down',
        x=1,
        xanchor='right',
        y=1.10
    )
]

fig.write_html(
    "Netflix_Interactive_Dashboard.html",
    include_plotlyjs=True,
    full_html=True
)

print("Dashboard exported successfully!")

from google.colab import files

files.download("Netflix_Interactive_Dashboard.html")

Dashboard exported successfully!


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>